# Module 2 HW

Patrick Maloon

kvq3jb

In [1]:
import pandas as pd

# Part 1:
### Converting Jane Austen's Sense and Sensibility from raw text into a dataframe of tokens.

In [6]:
text_file = 'pg161.txt'

OHCO = ['chap_num', 'para_num', 'sent_num', 'token_num']

epub = open(text_file, 'r', encoding='utf-8-sig').readlines()
df = pd.DataFrame(epub, columns=['line_str'])
df.index.name = 'line_num'
df.line_str = df.line_str.str.strip()
df.head()

,line_str
line_num,
0,﻿The Project Gutenberg EBook of Sense and Sens...
1,
2,This eBook is for the use of anyone anywhere a...
3,almost no restrictions whatsoever. You may co...
4,re-use it under the terms of the Project Guten...


In [7]:
title = df.loc[0].line_str.replace('The Project Gutenberg EBook of ', '')
df['Title'] = title
print(title)
df.head()

Sense and Sensibility, by Jane Austen


,line_str,Title
line_num,,
0,﻿The Project Gutenberg EBook of Sense and Sens...,"﻿Sense and Sensibility, by Jane Austen"
1,,"﻿Sense and Sensibility, by Jane Austen"
2,This eBook is for the use of anyone anywhere a...,"﻿Sense and Sensibility, by Jane Austen"
3,almost no restrictions whatsoever. You may co...,"﻿Sense and Sensibility, by Jane Austen"
4,re-use it under the terms of the Project Guten...,"﻿Sense and Sensibility, by Jane Austen"


In [8]:
a = df.line_str.str.match(r"\*\*\*\s*START OF (THE|THIS) PROJECT")
b = df.line_str.str.match(r"\*\*\*\s*END OF (THE|THIS) PROJECT")

an = df.loc[a].index[0]
bn = df.loc[b].index[0]
df = df.loc[an + 1 : bn - 2]
df.head()

,line_str,Title
line_num,,
19,,"﻿Sense and Sensibility, by Jane Austen"
20,Special thanks are due to Sharon Partridge for...,"﻿Sense and Sensibility, by Jane Austen"
21,proofreading and correction of this etext.,"﻿Sense and Sensibility, by Jane Austen"
22,,"﻿Sense and Sensibility, by Jane Austen"
23,,"﻿Sense and Sensibility, by Jane Austen"


In [9]:
chap_lines = df.line_str.str.match(r"^\s*(chapter|letter)\s+(\d+)", case=False)
chap_nums = [i+1 for i in range(df.loc[chap_lines].shape[0])]
df.loc[chap_lines, 'chap_num'] = chap_nums
df.chap_num = df.chap_num.ffill()

df = df.loc[~df.chap_num.isna()]
df = df.loc[~chap_lines]           
df.chap_num = df.chap_num.astype('int')
df.head()

,line_str,Title,chap_num
line_num,,,
43,,"﻿Sense and Sensibility, by Jane Austen",1
44,,"﻿Sense and Sensibility, by Jane Austen",1
45,The family of Dashwood had long been settled i...,"﻿Sense and Sensibility, by Jane Austen",1
46,"was large, and their residence was at Norland ...","﻿Sense and Sensibility, by Jane Austen",1
47,"their property, where, for many generations, t...","﻿Sense and Sensibility, by Jane Austen",1


In [ ]:
dfc = df.groupby(OHCO[:1]).line_str.apply(lambda x: '\n'.join(x)).to_frame()
dfc.head()

,line_str
chap_num,
1,\n\nThe family of Dashwood had long been settl...
2,\n\nMrs. John Dashwood now installed herself m...
3,\n\nMrs. Dashwood remained at Norland several ...
4,"\n\n""What a pity it is, Elinor,"" said Marianne..."
5,"\n\nNo sooner was her answer dispatched, than ..."


In [11]:
dfp = dfc['line_str'].str.split(r'\n\n+', expand=True).stack()\
    .to_frame().rename(columns={0:'para_str'})
dfp.index.names = OHCO[:2]
dfp['para_str'] = dfp['para_str'].str.replace(r'\n+', ' ').str.strip()
dfp = dfp[~dfp['para_str'].str.match(r'^\s*$')]
dfp.head() 

para_str
chap_num para_num                                                   
1        1         The family of Dashwood had long been settled i...
         2         By a former marriage, Mr. Henry Dashwood had o...
         3         The old gentleman died: his will was read, and...
         4         Mr. Dashwood's disappointment was, at first, s...
         5         His son was sent for as soon as his danger was...

In [12]:
dfs = dfp['para_str'].str.split(r'[.?!;:"]+', expand=True).stack()\
    .to_frame().rename(columns={0:'sent_str'})
dfs.index.names = OHCO[:3]
dfs = dfs[~dfs['sent_str'].str.match(r'^\s*$')]
dfs.head()

sent_str
chap_num para_num sent_num                                                   
1        1        0         The family of Dashwood had long been settled i...
                  1           Their estate\nwas large, and their residence...
                  2           The late owner of this estate was a single\n...
                  3           But her\ndeath, which happened ten years bef...
                  4          for to supply her loss, he invited and receiv...

In [19]:
dft = dfs['sent_str'].str.split(r"[\s',-]+", expand=True).stack()\
    .to_frame().rename(columns={0:'token_str'})
dft.index.names = OHCO[:4]
dft.head(25)

token_str
chap_num para_num sent_num token_num           
1        1        0        0                The
                           1             family
                           2                 of
                           3           Dashwood
                           4                had
                           5               long
                           6               been
                           7            settled
                           8                 in
                           9             Sussex
                  1        0                   
                           1              Their
                           2             estate
                           3                was
                           4              large
                           5                and
                           6              their
                           7          residence
                           8                was
                           9                 at
                           10           Norland
                           11              Park
                           12                in
                           13               the
                           14            centre

In [20]:
sentences = dft.groupby(OHCO[:3]).token_str.apply(lambda x: ' '.join(x)).to_frame().rename(columns={'token_str':'content'})
paragraphs = dft.groupby(OHCO[:2]).token_str.apply(lambda x: ' '.join(x)).to_frame().rename(columns={'token_str':'content'})
chapters = dft.groupby(OHCO[:1]).token_str.apply(lambda x: ' '.join(x)).to_frame().rename(columns={'token_str':'content'})

def gather(ohco_level):
    return df.groupby(OHCO[:ohco_level]).token_str\
        .apply(lambda x: ' '.join(x))\
        .to_frame()\
        .rename(columns={'token_str':'content'})

In [21]:
sentences.head()

content
chap_num para_num sent_num                                                   
1        1        0         The family of Dashwood had long been settled i...
                  1          Their estate was large and their residence wa...
                  2          The late owner of this estate was a single ma...
                  3          But her death which happened ten years before...
                  4          for to supply her loss he invited and receive...

In [22]:
dft.to_csv('Sense_and_Sensibility.csv')

# Part 2:
### combining both Persuasion and Sense and Sensibility into a single dataframe with an appropriately modified OHCO list.

In [29]:
OHCO_combined = ['book_title', 'chap_num', 'para_num', 'sent_num', 'token_num']

df_persuasion = pd.read_csv('austen-persuasion.csv', index_col=OHCO)
df_Sense = pd.read_csv('Sense_and_Sensibility.csv', index_col=OHCO)

In [30]:
df_persuasion['book_title'] = 'Persuasion'
df_persuasion_indexed = df_persuasion.reset_index()
df_persuasion_indexed = df_persuasion_indexed.set_index(OHCO_combined)

In [31]:
df_combined = pd.concat([df_persuasion_indexed, df_Sense])
df_combined.to_csv('Austen_Persuasion_Sense_and_Sensibility.csv')

In [32]:
df_combined.head(25)

,token_str
"(Persuasion, 1, 1, 0, 0)",Sir
"(Persuasion, 1, 1, 0, 1)",Walter
"(Persuasion, 1, 1, 0, 2)",Elliot
"(Persuasion, 1, 1, 0, 3)",of
"(Persuasion, 1, 1, 0, 4)",Kellynch
"(Persuasion, 1, 1, 0, 5)",Hall
"(Persuasion, 1, 1, 0, 6)",in
"(Persuasion, 1, 1, 0, 7)",Somersetshire
"(Persuasion, 1, 1, 0, 8)",was
"(Persuasion, 1, 1, 0, 9)",a


In [36]:
df_combined.sample(50)

,token_str
"(4, 11, 3, 3)",required
"(33, 46, 1, 12)",the
"(4, 20, 10, 9)",far
"(28, 7, 0, 9)",both
"(Persuasion, 21, 32, 11, 11)",a
"(10, 20, 7, 3)",has
"(Persuasion, 12, 61, 4, 18)",think
"(Persuasion, 22, 21, 3, 6)",of
"(17, 1, 6, 7)",before
"(38, 22, 4, 5)",at
